# Alpha-Seeker — End-to-end pipeline

**CS 410 — Text Information Systems (Spring 2026)**

This notebook runs the full Reddit → preprocess → VADER sentiment → LDA topic pipeline used in the Alpha-Seeker project. Run all cells top-to-bottom (`Run All`).

**Prerequisites:** Python 3.10+ recommended. From the project root (`CS410-FinalProject`):

```bash
pip install -r requirements.txt
```

**Outputs:**
- `data/processed/reddit_posts_cleaned.csv` — tokenized text for LDA
- `data/processed/reddit_posts_sentiment.csv`, `data/processed/sentiment_daily.csv`, `public/sentiment_summary.json` — VADER outputs
- `data/processed/lda_topic_assignments.csv`, `data/processed/lda_results.json`, `public/lda_summary.json` — topic model + dashboard JSON

The React dashboard (`npm run dev`) reads `public/sentiment_summary.json` and `public/lda_summary.json`.

## 1. Project root & imports

Ensures `src/*.py` modules resolve the same paths as when run from the CLI (`data/…`, `public/…`).

In [ ]:
import os
import sys
from pathlib import Path

# Directory containing this notebook
NB_DIR = Path.cwd().resolve()
# Project root: parent of notebooks/ if we are in notebooks/, else cwd
if NB_DIR.name == "notebooks":
    PROJECT_ROOT = NB_DIR.parent
else:
    PROJECT_ROOT = NB_DIR

os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")

## 2. (Optional) Fresh Reddit collection

The Arctic Shift API needs network access and **many** rate-limited requests (on the order of **hours** for the default date range and keyword grid).

- Set `RUN_COLLECTION = False` to use existing `data/raw/reddit_posts_apr.csv`.
- Set `RUN_COLLECTION = True` only when you intentionally want to refresh raw data.

In [ ]:
RUN_COLLECTION = False  # set True to re-fetch from Arctic Shift (slow)

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "reddit_posts_apr.csv"

if RUN_COLLECTION:
    import reddit_collector as rc

    posts = rc.collect_all_posts()
    rc.save_to_csv(posts, str(RAW_CSV))
    rc.print_summary(posts)
    print(f"\nSaved raw posts to {RAW_CSV}")
else:
    if not RAW_CSV.is_file():
        raise FileNotFoundError(
            f"Expected bundled raw data at {RAW_CSV}. "
            "Set RUN_COLLECTION = True to collect, or add the CSV under data/raw/."
        )
    print(f"Using existing raw data ({RAW_CSV.stat().st_size / 1024 / 1024:.2f} MB).")

## 3. Preprocessing (raw → cleaned)

Same cleaning rules as `notebooks/reddit_eda.ipynb`, implemented in `src/preprocess_from_raw.py`.

In [ ]:
import preprocess_from_raw as pfr

pfr.main()

## 4. Sentiment (VADER)

Reads `data/processed/reddit_posts_cleaned.csv`, writes scored CSV, daily aggregates, and `public/sentiment_summary.json`.

In [ ]:
import sentiment

sentiment.main()

## 5. LDA topic modeling

Grid search over *k*, coherence scoring, topic assignments, paper-ready JSON, and `public/lda_summary.json` for the dashboard. **This step can take several minutes.**

In [ ]:
import lda_analysis

lda_analysis.main()

## 6. Done

Pipeline complete. Optional next steps:
- `notebooks/reddit_eda.ipynb` — figures and extra EDA
- `npm install && npm run dev` — Kalshi market search UI (separate from this Python pipeline)